# CSI 300 parallel minute-cache generator

Read raw market pickle files, build market partitions, merge one complete `basket_tick_v03` cache per trading day, and optionally remove the partitions after validation.


In [1]:
import importlib
from pathlib import Path
import sys

_root_candidates = [Path.cwd().resolve(), Path.cwd().resolve() / "Stock-Index-Fitting"]
PROJECT_ROOT = next(
    (candidate for candidate in _root_candidates if (candidate / "utils" / "cache_generator.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Cannot locate Stock-Index-Fitting from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
from xtquant import xtdata

import utils.minute_tick_cache_v03 as minute_tick_cache_module
minute_tick_cache_module = importlib.reload(minute_tick_cache_module)
import utils.cache_generator as cache_generator_module
cache_generator_module = importlib.reload(cache_generator_module)

from utils.cache_generator import (
    CSI300_INDEX_CODE,
    generate_csi300_caches,
    merge_partition_caches_for_range,
    preview_generation_plan,
    resolve_trading_dates,
    select_complete_trade_dates,
)


## Configuration

`START_DATE` and `END_DATE` control both partition generation and complete-cache merging. Non-trading endpoints are moved to the closest previous SH trading day through the XtQuant calendar.


In [2]:
INDEX_CODE = CSI300_INDEX_CODE  # Fixed to CSI 300: 000300
START_DATE = "20260501"
END_DATE = "20260630"
XT_PORT = 58610
MAX_WORKERS = 4
FORCE_REBUILD = False

# Delete validated market partition pickle files after a complete cache exists.
DELETE_PARTITION_CACHES = True

# Rebuild an already valid complete cache during the merge phase.
OVERWRITE_COMPLETE_CACHE = False

SOURCE_TICK_ROOT = Path(r"\\192.168.1.138\康曼德共享\高频行情迅投\ticks")
WEIGHTS_DIR = PROJECT_ROOT / "data" / "weights_projection"
CACHE_DIR = PROJECT_ROOT / "data" / INDEX_CODE / "_tick_cache_correlation_v03"

print("Project root:", PROJECT_ROOT)
print("Source tick root:", SOURCE_TICK_ROOT)
print("Weights directory:", WEIGHTS_DIR)
print("Cache directory:", CACHE_DIR)
print("Workers:", MAX_WORKERS)
print("Force rebuild:", FORCE_REBUILD)
print("Delete partition caches:", DELETE_PARTITION_CACHES)


Project root: E:\Codex\系统\Stock-Index-Fitting
Source tick root: \\192.168.1.138\康曼德共享\高频行情迅投\ticks
Weights directory: E:\Codex\系统\Stock-Index-Fitting\data\weights_projection
Cache directory: E:\Codex\系统\Stock-Index-Fitting\data\000300\_tick_cache_correlation_v03
Workers: 4
Force rebuild: False
Delete partition caches: True


## Resolve trading dates


In [3]:
xtdata.reconnect(port=XT_PORT)
date_resolution = resolve_trading_dates(
    START_DATE,
    END_DATE,
    xtdata_client=xtdata,
)
trade_dates = list(date_resolution.trade_dates)

print("Configured range:", date_resolution.configured_start_date, date_resolution.configured_end_date)
print("Adjusted range:  ", date_resolution.adjusted_start_date, date_resolution.adjusted_end_date)
print(f"Trading dates ({len(trade_dates)}):", trade_dates)


***** xtdata连接成功 2026-07-30 16:52:55*****
服务信息: {'tag': 'qmt_research', 'version': '1.0'}
服务地址: 127.0.0.1:58610
数据路径: E:\迅投极速交易终端睿智融科版\datadir
设置xtdata.enable_hello = False可隐藏此消息

Configured range: 20260501 20260630
Adjusted range:   20260430 20260630
Trading dates (40): ['20260430', '20260506', '20260507', '20260508', '20260511', '20260512', '20260513', '20260514', '20260515', '20260518', '20260519', '20260520', '20260521', '20260522', '20260525', '20260526', '20260527', '20260528', '20260529', '20260601', '20260602', '20260603', '20260604', '20260605', '20260608', '20260609', '20260610', '20260611', '20260612', '20260615', '20260616', '20260617', '20260618', '20260622', '20260623', '20260624', '20260625', '20260626', '20260629', '20260630']


## Preview inputs

This metadata-only check does not deserialize the large source pickle files.


In [4]:
plan = preview_generation_plan(
    trade_dates,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
)
display(plan)

preview_complete_trade_dates, excluded_trade_date_reasons = select_complete_trade_dates(
    plan,
    trade_dates,
)

if excluded_trade_date_reasons:
    print("\nExcluded incomplete dates:")
    for excluded_date, reasons in excluded_trade_date_reasons.items():
        print(f"  {excluded_date}")
        for reason in reasons:
            print(f"    - {reason}")

print(f"\nDates with complete preview inputs ({len(preview_complete_trade_dates)}):", preview_complete_trade_dates)
print(f"Dates scheduled for cache generation or unavailable marking ({len(trade_dates)}):", trade_dates)
if not trade_dates:
    raise RuntimeError("No trading dates are available for cache generation.")


,trade_date,weight_file,component_count,market,market_stock_count,source_path,source_exists,source_gb,final_cache_path,final_cache_exists
0,20260430,沪深300_样本权重_20260331.csv,300,sh_kcb,19,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\04\...,True,0.692,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,True
1,20260430,沪深300_样本权重_20260331.csv,300,sh_zb,171,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\04\...,True,2.087,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,True
2,20260430,沪深300_样本权重_20260331.csv,300,sz_cyb,35,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\04\...,True,1.503,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,True
3,20260430,沪深300_样本权重_20260331.csv,300,sz_zb,75,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\04\...,True,1.713,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,True
4,20260506,沪深300_样本权重_20260430.csv,300,sh_kcb,19,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\05\...,True,0.711,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
...,...,...,...,...,...,...,...,...,...,...
155,20260629,沪深300_样本权重_20260529.csv,300,sz_zb,75,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\06\...,True,1.722,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
156,20260630,沪深300_样本权重_20260529.csv,300,sh_kcb,19,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\06\...,True,0.726,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
157,20260630,沪深300_样本权重_20260529.csv,300,sh_zb,171,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SH\2026\06\...,True,2.066,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False
158,20260630,沪深300_样本权重_20260529.csv,300,sz_cyb,35,\\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\06\...,True,1.557,E:\Codex\系统\Stock-Index-Fitting\data\000300\_t...,False



Excluded incomplete dates:
  20260603
    - Missing source pickle: \\192.168.1.138\康曼德共享\高频行情迅投\ticks\SZ\2026\06\20260603_tick_sz_zb.pkl

Dates with complete preview inputs (39): ['20260430', '20260506', '20260507', '20260508', '20260511', '20260512', '20260513', '20260514', '20260515', '20260518', '20260519', '20260520', '20260521', '20260522', '20260525', '20260526', '20260527', '20260528', '20260529', '20260601', '20260602', '20260604', '20260605', '20260608', '20260609', '20260610', '20260611', '20260612', '20260615', '20260616', '20260617', '20260618', '20260622', '20260623', '20260624', '20260625', '20260626', '20260629', '20260630']
Dates scheduled for cache generation or unavailable marking (40): ['20260430', '20260506', '20260507', '20260508', '20260511', '20260512', '20260513', '20260514', '20260515', '20260518', '20260519', '20260520', '20260521', '20260522', '20260525', '20260526', '20260527', '20260528', '20260529', '20260601', '20260602', '20260603', '20260604', '2026060

## Generate market partitions and complete caches

Dates are processed sequentially. Physical market files within one date are handled by the configured process pool.


In [ ]:
generated_cache_paths = generate_csi300_caches(
    trade_dates,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
    max_workers=MAX_WORKERS,
    force_rebuild=FORCE_REBUILD,
)

print(f"Complete caches available after generation: {len(generated_cache_paths)}")
for cache_path in generated_cache_paths:
    print(" ", cache_path)


[DATE 1/40] 20260430: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260331.csv, components=300
  [final:all] cache_hit: basket_minute_wide_20260430_445b76ac98d5.pkl
[DATE 2/40] 20260506: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260430.csv, components=300
  [partition:sh_kcb] built (26.6s)
  [partition:sz_cyb] built (52.3s)
  [partition:sz_zb] built (56.9s)
  [partition:sh_zb] built (62.2s)
  [final:all] built (64.1s): basket_minute_wide_20260506_be199932c155.pkl
[DATE 3/40] 20260507: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260430.csv, components=300
  [partition:sh_kcb] built (27.0s)
  [partition:sz_cyb] built (51.8s)
  [partition:sz_zb] built (56.8s)
  [partition:sh_zb] built (61.7s)
  [final:all] built (64.5s): basket_minute_wide_20260507_a9c15561ad45.pkl
[DATE 4/40] 20260508: preparing CSI 300 cache with max_workers=4
  weights=沪深300_样本权重_20260430.csv, components=300
  [partition:sh_kcb] built (27.7s)
  [partitio

In [ ]:
generated_cache_paths = generate_csi300_caches(
    trade_dates,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
    max_workers=MAX_WORKERS,
    force_rebuild=FORCE_REBUILD,
)

print(f"Complete caches available after generation: {len(generated_cache_paths)}")
for cache_path in generated_cache_paths:
    print(" ", cache_path)


## Merge, validate, and clean partitions

The merge range uses the same `START_DATE` and `END_DATE`. Partition files are deleted only after the complete cache passes schema, date, stock-universe, missing-stock, and minute-row validation.


In [ ]:
complete_cache_paths = merge_partition_caches_for_range(
    START_DATE,
    END_DATE,
    xtdata_client=xtdata,
    weights_dir=WEIGHTS_DIR,
    source_tick_root=SOURCE_TICK_ROOT,
    cache_dir=CACHE_DIR,
    delete_partition_caches=DELETE_PARTITION_CACHES,
    overwrite_complete_cache=OVERWRITE_COMPLETE_CACHE,
)

print(f"Validated complete caches: {len(complete_cache_paths)}")
for cache_path in complete_cache_paths:
    print(" ", cache_path)
